# Chapitre 4 — Sélectionner, filtrer, trier

## Le problème

Sur 271 116 participations olympiques, on veut les footballeuses, les médaillés belges, les épreuves d'une année donnée. Il faut savoir désigner précisément un sous-ensemble de lignes et de colonnes.

En SQL, on écrit :

```sql
SELECT Name, Year, Medal
FROM jo
WHERE Sport = 'Football' AND Sex = 'F'
ORDER BY Year;
```

Ce chapitre construit l'équivalent Pandas pièce par pièce. Il repose sur une distinction essentielle : **sélectionner par étiquette** (`loc`) ou **par position** (`iloc`).

In [ ]:
import numpy as np
import pandas as pd

jo = pd.read_csv("../donnees/bstorm/Exercices_Pandas_athlete_events.csv")
s = pd.Series([0.25, 0.5, 0.75, 1.0], index=["a", "b", "c", "d"])

## 1. Sélectionner des colonnes

### Une colonne : une Series

In [ ]:
jo["Name"].head(3)

In [ ]:
type(jo["Name"])

La notation pointée `jo.Name` fonctionne aussi, mais seulement si le nom de colonne est un identifiant Python valide et n'entre pas en conflit avec une méthode. Une colonne `count` ou `"Shooting Accuracy"` ne peut pas s'écrire ainsi. Les crochets fonctionnent toujours.

### Plusieurs colonnes : un DataFrame

In [ ]:
jo[["Name", "Year", "Medal"]].head(3)

> **Comparaison.** Quelle différence entre `jo["Name"]` et `jo[["Name"]]` ? Vérifiez le type et la forme de chacun.

In [ ]:
type(jo[["Name"]]), jo["Name"].shape, jo[["Name"]].shape

Les doubles crochets passent une **liste** de noms : le résultat est toujours un DataFrame, même avec un seul nom. Certaines fonctions attendent une table (par exemple les modèles de scikit-learn pour les variables explicatives) : il faut alors `df[["col"]]`.

## 2. `loc` et `iloc` : étiquettes ou positions

| | `loc` | `iloc` |
|---|---|---|
| Désigne par | étiquette (index, nom de colonne) | position entière (0, 1, 2…) |
| Tranche `a:b` | **inclut** `b` | **exclut** `b` (comme Python) |
| Exemple | `df.loc[10, "Name"]` | `df.iloc[10, 1]` |

La syntaxe est toujours `[lignes, colonnes]`, comme pour un tableau NumPy à deux dimensions.

### Sur une Series

In [ ]:
s

> **Prédiction.** Que renvoient `s.loc["a":"c"]` et `s.iloc[0:2]` ? Combien de valeurs chacun ?

In [ ]:
s.loc["a":"c"]

In [ ]:
s.iloc[0:2]

Avec `loc`, la borne de fin est **incluse** : demander les étiquettes de `a` à `c`, c'est attendre `c` dans le résultat. Avec `iloc`, on retrouve la convention de Python et de NumPy : la fin est exclue.

Les pas négatifs fonctionnent comme sur une liste :

In [ ]:
s.iloc[::-1]

### Les crochets seuls sur une Series : à éviter

En Pandas 3, `s["a"]` désigne toujours une **étiquette**. Une ancienne habitude consistait à écrire `s[0]` pour la première valeur : cela lève désormais une erreur quand l'index n'est pas composé d'entiers.

In [ ]:
s[0]

In [ ]:
s.iloc[0], s.loc["a"]

Avec un index d'entiers, `s[0]` cherche l'**étiquette** 0, pas la première position, ce qui devient trompeur après un tri. Dans ce cours, on écrit toujours `loc` ou `iloc`.

### Sur un DataFrame

In [ ]:
jo.loc[0:4, ["Name", "Sport", "Year"]]       # étiquettes 0 à 4 incluses : 5 lignes

In [ ]:
jo.iloc[0:4, [1, 12, 9]]                     # positions 0 à 3 : 4 lignes

In [ ]:
jo.loc[:, "Name":"Weight"].head(2)           # tranche de colonnes par nom, bornes incluses

### Ce que renvoie la sélection

| Sélection | Résultat |
|---|---|
| une ligne, une colonne : `df.loc[0, "Name"]` | une valeur (scalaire) |
| une ligne, plusieurs colonnes : `df.loc[0, ["Name", "Age"]]` | une Series |
| plusieurs lignes, une colonne : `df.loc[0:3, "Name"]` | une Series |
| plusieurs lignes, plusieurs colonnes | un DataFrame |

Pour lire ou écrire **une seule valeur**, `at` (étiquette) et `iat` (position) sont plus rapides :

In [ ]:
jo.at[0, "Name"], jo.iat[0, 1]

## 3. Filtrer avec un masque booléen

Une comparaison sur une colonne produit une Series de booléens : un **masque**.

In [ ]:
masque = jo["Sport"] == "Football"
masque.head()

In [ ]:
masque.sum()          # True compte pour 1 : nombre de lignes concernées

In [ ]:
jo[masque].head(3)

On peut écrire le masque directement entre les crochets, ou dans `loc` pour choisir en même temps les colonnes. C'est l'équivalent exact de `SELECT ... WHERE` :

In [ ]:
jo.loc[jo["Sport"] == "Football", ["Name", "Team", "Year"]].head(3)

### Combiner plusieurs conditions

| Opérateur | Sens | SQL |
|---|---|---|
| `&` | et | `AND` |
| `\|` | ou | `OR` |
| `~` | non | `NOT` |

Chaque condition doit être **entre parenthèses**, car `&` et `|` sont prioritaires sur `==` en Python.

In [ ]:
footballeuses = jo[(jo["Sport"] == "Football") & (jo["Sex"] == "F")]
footballeuses.shape

> **Trouver l'erreur.** La cellule suivante devrait sélectionner les médaillés de 2016. Elle échoue. Pourquoi ?

In [ ]:
jo[(jo["Year"] == 2016) and (jo["Medal"].notna())]

Le mot-clé `and` de Python demande « cette Series est-elle vraie ? », ce qui n'a pas de sens pour 271 116 valeurs. Pandas refuse de répondre et lève `ValueError: The truth value of a Series is ambiguous`. Il faut l'opérateur élément par élément `&`.

In [ ]:
medailles_2016 = jo[(jo["Year"] == 2016) & (jo["Medal"].notna())]
medailles_2016.shape

> **Trouver l'erreur.** Et celle-ci ? Elle ne lève pas d'erreur, mais le résultat est faux.

In [ ]:
jo[jo["Year"] == 2016 & jo["Medal"].notna()].shape

Sans parenthèses, Python évalue d'abord `2016 & jo["Medal"].notna()`, puis compare `Year` au résultat. Selon les cas, on obtient une erreur ou un filtre qui ne fait pas ce qu'on croit. Les parenthèses ne sont jamais facultatives.

### Appartenance à une liste : `isin`

L'équivalent de `IN (...)` en SQL.

In [ ]:
jo[jo["NOC"].isin(["BEL", "NED", "LUX"])]["NOC"].value_counts()

La fonction NumPy `np.isin` fait la même chose sur un tableau et renvoie un tableau de booléens, sans index :

In [ ]:
np.isin(jo["NOC"].to_numpy()[:5], ["CHN", "DEN"])

### Intervalle : `between`

L'équivalent de `BETWEEN`. Les deux bornes sont incluses par défaut (`inclusive="both"`).

In [ ]:
jo[jo["Year"].between(1984, 2016)]["Year"].agg(["min", "max"])

### Négation : `~`

In [ ]:
jo[~jo["Season"].eq("Summer")]["Season"].unique()

### `query` : une syntaxe proche de SQL

`query` reçoit la condition sous forme de texte. Les noms de colonnes s'écrivent directement, et une variable Python se préfixe par `@`.

In [ ]:
annee = 2016
jo.query("Sport == 'Football' and Sex == 'F' and Year == @annee")[["Name", "Team", "Medal"]].head()

Dans `query`, `and` et `or` sont acceptés : la chaîne est analysée par Pandas, pas par Python. C'est souvent plus lisible pour des conditions longues ; les masques restent plus souples (on peut les construire, les nommer, les réutiliser).

## 4. Masques et alignement

Un masque est lui-même une Series : il est **aligné sur l'index**.

In [ ]:
s2 = pd.Series(np.arange(3), index=list("cde"))
s2 > 0

> **Prédiction.** Que va produire `s.loc[s2 > 0]` ? Le masque a 3 valeurs, `s` en a 4.

In [ ]:
s.loc[s2 > 0]

Pandas cherche, pour chaque étiquette de `s`, la valeur du masque. Les étiquettes `a` et `b` n'existent pas dans le masque, et l'étiquette `e` du masque n'existe pas dans `s` : l'alignement est impossible.

Les règles pratiques :

- `s[s > 0.5]` ou `s.loc[s > 0.5]` : le masque est calculé sur la même Series, donc aligné ;
- `s.iloc[masque.to_numpy()]` : on passe un tableau brut de même longueur, sans aucun alignement.

## 5. Choisir un index significatif

Par défaut, l'index est une suite d'entiers. On peut le remplacer par une colonne qui identifie les lignes, ce qui rend `loc` plus parlant.

In [ ]:
produits = pd.read_csv("../donnees/demo/produits.csv")
produits = produits.set_index("produit_id")
produits.head(3)

In [ ]:
produits.loc["P03", "prix"]

In [ ]:
produits.index.name, produits.index.is_unique

`reset_index()` fait l'opération inverse : l'index redevient une colonne.

In [ ]:
produits.reset_index().head(2)

### Trancher un index trié

Une tranche par étiquette suppose que l'index soit **trié**. Exemple : compter les participations par année, puis sélectionner une période.

In [ ]:
par_annee = jo["Year"].value_counts()
par_annee.head()

> **Prédiction.** `par_annee` est trié par effectif décroissant, pas par année. Que va donner `par_annee.loc[1984:2016]` ?

In [ ]:
par_annee.loc[1984:2016]

La tranche suit l'**ordre des lignes** : elle part de la position de l'étiquette 1984 et s'arrête à la position de l'étiquette 2016. Ici, 2016 se trouve *avant* 1984 dans la Series : la tranche est vide. Avec d'autres bornes, on obtiendrait un mélange arbitraire d'années, sans aucun avertissement. Il faut trier l'index d'abord :

In [ ]:
par_annee.sort_index().loc[1984:2016]

## 6. Trier

`sort_values` trie selon une ou plusieurs colonnes. C'est l'équivalent de `ORDER BY`.

In [ ]:
produits.sort_values("prix", ascending=False).head(3)

In [ ]:
jo.sort_values(["Year", "Age"], ascending=[False, True])[["Year", "Age", "Name", "Sport"]].head()

`sort_index()` trie selon l'index. `nlargest` et `nsmallest` donnent directement les extrêmes, sans trier toute la table :

In [ ]:
produits.nlargest(3, "prix")

## 7. Modifier les lignes sélectionnées

`loc` permet aussi d'**écrire** dans une sélection : `df.loc[masque, colonne] = valeur`.

In [ ]:
produits.loc[produits["categorie"] == "Mobilier", "prix"] = produits["prix"] * 0.9   # remise de 10 %
produits[produits["categorie"] == "Mobilier"]

Le chapitre 5 explique pourquoi cette écriture en une seule opération `loc[...] = ...` est la seule fiable, et pourquoi `df[masque]["prix"] = ...` ne modifie rien.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| `and`, `or`, `not` sur des Series | `ValueError: truth value ... ambiguous` | `&`, `\|`, `~` |
| Conditions sans parenthèses | erreur ou filtre faux | `(cond1) & (cond2)` |
| `s[0]` sur un index de texte | `KeyError` en Pandas 3 | `s.iloc[0]` |
| Tranche par étiquette sur un index non trié | résultat incohérent | `sort_index()` d'abord |
| `loc[a:b]` vs `iloc[a:b]` | une ligne de plus ou de moins | `loc` inclut la fin, `iloc` l'exclut |
| Masque issu d'une autre table | `IndexingError: Unalignable boolean Series` | calculer le masque sur la même table, ou `.to_numpy()` |

## Exercices

- Exercice Bstorm **JO**, questions **9 à 16**. La colonne « Edition » de l'énoncé s'appelle `Games` dans le fichier.
- Exercice supplémentaire **S4** : corriger des filtres défectueux.
- Renforcement : drills **D02 Euro12**, **D02 Fictional Army**, **D02 Chipotle**, et **D10 Alcohol counter** pour approfondir l'index.

## À retenir

- `df["col"]` donne une Series, `df[["col"]]` un DataFrame.
- `loc` = étiquettes, fin incluse ; `iloc` = positions, fin exclue ; `at`/`iat` pour une valeur.
- Un filtre = un masque booléen, combiné avec `&`, `|`, `~` et des parenthèses.
- `isin`, `between`, `query` couvrent `IN`, `BETWEEN` et les `WHERE` lisibles.
- `set_index` donne un index significatif ; trier l'index avant de trancher par étiquette.
- `sort_values`, `sort_index`, `nlargest` pour l'ordre.